In [12]:
## collecte des données du sp500
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from fredapi import Fred


fred = Fred(api_key = "d74040a70c15768277dabddda3a0fe47" )
tables = pd.read_html("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",storage_options={"User-Agent": "Mozilla/5.0"})

ticker_sp500_df = tables[0]["Symbol"].tolist()

dividends = {}
for ticker in ticker_sp500_df:
    series = yf.Ticker(ticker).dividends
    if isinstance(series.index, pd.DatetimeIndex) and series.index.tz is not None:
        series = series.tz_localize(None)
        
    dividends[ticker] = series.loc["2014-01-01":"2024-12-31"]

Dividends = pd.DataFrame(dividends)

shares= {}
for ticker in ticker_sp500_df :
    series = yf.Ticker(ticker).get_shares_full(start="2014-01-01", end="2024-12-31")
    if series is None or series.empty:
        continue
    if isinstance(series.index,pd.DatetimeIndex) and series.index.tz is not None :
        series = series.tz_localize(None)
    
    series = series.groupby(level=0).last()  # garde la valeur la plus récente en cas de doublons
    shares[ticker] = series.loc["2014-01-01":"2024-12-31"]

shares = pd.DataFrame(shares)   
shares.index.name = "Date" 

sp500_prices = yf.download(ticker_sp500_df, start = "2014-01-01", end = "2024-12-31", auto_adjust=False)
vix_prices = yf.download("^VIX", start = "2014-01-01", end = "2024-12-31")
rate_10y = fred.get_series("DGS10", observation_start="2014-01-01",observation_end="2024-12-31")
rate_2y = fred.get_series("DGS2", observation_start="2014-01-01",observation_end="2024-12-31")


$BRK.B: possibly delisted; no timezone found
$BF.B: possibly delisted; no price data found  (1d 1927-05-04 -> 2026-04-09)
[*********             19%                       ]  94 of 503 completed$SNDK: possibly delisted; no price data found  (1d 2014-01-01 -> 2024-12-31) (Yahoo error = "Data doesn't exist for startDate = 1388552400, endDate = 1735621200")
[**********            21%                       ]  108 of 503 completed$Q: possibly delisted; no price data found  (1d 2014-01-01 -> 2024-12-31) (Yahoo error = "Data doesn't exist for startDate = 1388552400, endDate = 1735621200")
[**********************61%****                   ]  306 of 503 completed$BF.B: possibly delisted; no price data found  (1d 2014-01-01 -> 2024-12-31)
[**********************89%******************     ]  449 of 503 completed$BRK.B: possibly delisted; no timezone found
[*********************100%***********************]  503 of 503 completed

4 Failed downloads:
['SNDK', 'Q']: possibly delisted; no price data foun

In [13]:

SP500_prices = sp500_prices["Adj Close"]
Volume = sp500_prices["Volume"]
VIX_prices = vix_prices["Close"]
VIX_prices.columns = ["VIX"]
yield_curve = rate_10y - rate_2y
Global_Rate = pd.DataFrame({"Rate_10Y": rate_10y, "Rate_2Y": rate_2y, "Yield_Curve": yield_curve})
SP500_prices.to_csv("data/raw/SP500_prices.csv")

shares.to_csv("data/raw/SP500_shares.csv")
Dividends.to_csv("data/raw/SP500_dividends.csv")
Volume.to_csv("data/raw/SP500_volume.csv")
VIX_prices.to_csv("data/raw/VIX_prices.csv")
Global_Rate.to_csv("data/raw/Global_Rate.csv")

print(SP500_prices.head())
print(VIX_prices.head())

print(type(SP500_prices))
print(SP500_prices.shape)
print(SP500_prices.columns[:5].tolist())
print(SP500_prices.head(3))

print(sp500_prices.columns.names)
print(sp500_prices.columns.get_level_values(0).unique())
print(sp500_prices.columns.get_level_values(1).unique()[:5])

Ticker              A       AAPL       ABBV  ABNB        ABT       ACGL  \
Date                                                                      
2014-01-02  36.303493  17.140667  31.909838   NaN  30.176613  18.184349   
2014-01-03  36.762054  16.764162  32.106289   NaN  30.500238  17.835686   
2014-01-06  36.581219  16.855566  30.933775   NaN  30.902790  17.667694   
2014-01-07  37.104347  16.735025  30.995155   NaN  30.665977  17.674034   
2014-01-08  37.711468  16.841003  30.915350   NaN  30.942266  17.569435   

Ticker            ACN       ADBE        ADI        ADM  ...         WY  \
Date                                                    ...              
2014-01-02  66.105850  59.290001  38.134869  30.258316  ...  19.549049   
2014-01-03  66.325859  59.160000  38.390236  30.399078  ...  19.592815   
2014-01-06  65.625130  58.119999  38.173561  30.462423  ...  19.392761   
2014-01-07  66.423630  58.970001  38.374763  30.159763  ...  19.449022   
2014-01-08  66.936966  58.9000

In [14]:
import pandas as pd

SP500 = pd.read_csv("data/raw/SP500_prices.csv", index_col="Date", parse_dates=True)
VIX   = pd.read_csv("data/raw/VIX_prices.csv",   index_col="Date", parse_dates=True)
rates = pd.read_csv("data/raw/Global_Rate.csv", index_col=0,parse_dates=True)

print(rates.head(3))

print(SP500.shape)
print(SP500.index[0])    # première date
print(SP500.index[-1])   # dernière date
print(SP500.head(3))

print(rates.head(10))    # regardez les NaN éventuels

            Rate_10Y  Rate_2Y  Yield_Curve
2014-01-01       NaN      NaN          NaN
2014-01-02      3.00     0.39         2.61
2014-01-03      3.01     0.41         2.60
(2767, 503)
2014-01-02 00:00:00
2024-12-30 00:00:00
                    A       AAPL       ABBV  ABNB        ABT       ACGL  \
Date                                                                      
2014-01-02  36.303493  17.140667  31.909838   NaN  30.176613  18.184349   
2014-01-03  36.762054  16.764162  32.106289   NaN  30.500238  17.835686   
2014-01-06  36.581219  16.855566  30.933775   NaN  30.902790  17.667694   

                  ACN       ADBE        ADI        ADM  ...         WY  \
Date                                                    ...              
2014-01-02  66.105850  59.290001  38.134869  30.258316  ...  19.549049   
2014-01-03  66.325859  59.160000  38.390236  30.399078  ...  19.592815   
2014-01-06  65.625130  58.119999  38.173561  30.462423  ...  19.392761   

                  WYNN       

In [15]:
shares = pd.read_csv("data/raw/SP500_shares.csv", index_col="Date", parse_dates=True)
print(shares.shape)
print(shares["AAPL"].dropna())


(2696, 498)
Date
2015-10-28    5.575330e+09
2016-01-28    5.544580e+09
2016-07-28    5.388440e+09
2016-10-26    5.332310e+09
2017-01-09    5.257820e+09
                  ...     
2024-11-02    1.511580e+10
2024-11-05    1.516960e+10
2024-11-16    1.511580e+10
2024-12-27    1.511580e+10
2024-12-28    1.511580e+10
Name: AAPL, Length: 284, dtype: float64
